In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchsummary import summary
import os


In [9]:
epochs = 15
batch_size = 32
learning_rate = 0.01

In [3]:
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
data_dir = "/content/drive/MyDrive/ALL DATASET/rooms_dataset"

In [5]:
dataset = datasets.ImageFolder(
    root = data_dir,
    transform = data_transform
)
num_class = len(dataset.classes)
print("total class: ", num_class)

total class:  3


In [6]:
train_size = int(0.8*len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print("Training images:", len(train_dataset))
print("Validation images:", len(test_dataset))


Training images: 94
Validation images: 24


In [7]:
# Data Loader

train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

In [10]:
# Load ResNet50 Model
model = efficientnet_b0(weights = EfficientNet_B0_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 86.8MB/s]


In [12]:
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_class)

In [13]:
for param in model.parameters():
  param.requires_grad = False

for param in model.classifier[1].parameters():
  param.requires_grad = True

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [15]:
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

In [16]:
from torch.autograd import no_grad
def train(model, loss_func, optimizer, train_loader, test_loader, epochs):
  for epoch in range(epochs):
    train_total = 0
    train_correct = 0
    running_loss = 0

    print(f"\n Epoch {epoch+1}/ {epochs}")
    print("="*30)

    for batch, (images, labels) in enumerate(train_loader):

      images = images.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)

      loss = loss_func(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()

      _,predict = torch.max(outputs, 1)
      train_total += labels.size(0)
      train_correct += (predict == labels).sum().item()

      train_accuracy = (train_correct / train_total)*100

    print(f"Training Loss: {running_loss:.4f}")
    print( f"Training Accuracy: {train_accuracy:.4f}")


    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
      for batch, (images, labels) in enumerate(test_loader):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        val_total += labels.size(0)
        val_correct += (predicted == labels).sum().item()

    val_accuracy = (val_correct / val_total)*100

    print(f"Validation Accuracy: {val_accuracy:.4f}")



In [17]:
train(model, loss_func, optimizer, train_loader, test_loader, epochs)


 Epoch 1/ 15
Training Loss: 3.1080
Training Accuracy: 48.9362
Validation Accuracy: 41.6667

 Epoch 2/ 15
Training Loss: 1.9829
Training Accuracy: 73.4043
Validation Accuracy: 66.6667

 Epoch 3/ 15
Training Loss: 0.9186
Training Accuracy: 93.6170
Validation Accuracy: 70.8333

 Epoch 4/ 15
Training Loss: 0.4344
Training Accuracy: 96.8085
Validation Accuracy: 83.3333

 Epoch 5/ 15
Training Loss: 0.2792
Training Accuracy: 98.9362
Validation Accuracy: 75.0000

 Epoch 6/ 15
Training Loss: 0.1887
Training Accuracy: 97.8723
Validation Accuracy: 75.0000

 Epoch 7/ 15
Training Loss: 0.1408
Training Accuracy: 98.9362
Validation Accuracy: 70.8333

 Epoch 8/ 15
Training Loss: 0.1010
Training Accuracy: 98.9362
Validation Accuracy: 75.0000

 Epoch 9/ 15
Training Loss: 0.0747
Training Accuracy: 98.9362
Validation Accuracy: 75.0000

 Epoch 10/ 15
Training Loss: 0.0750
Training Accuracy: 98.9362
Validation Accuracy: 75.0000

 Epoch 11/ 15
Training Loss: 0.0693
Training Accuracy: 98.9362
Validation Accu